# Chapter 3 &mdash; Why Per-Character Maps Are Always Safe

**Concept 11 of the Chapter 3 decomposition:** *The Crux of Homomorphisms: Why Character Maps Are Always Safe*

A multi-character mapping can <b>conflict on substrings</b>. A single-character one cannot &mdash; there are no substrings to conflict over.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter3-Star/Concept-Crux-Of-Homomorphisms/Concept-Crux-Of-Homomorphisms.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


When does a proposed mapping **fail** to be a homomorphism? When it special-cases.

Let $h(\text{Hel})=\text{ooo}$, $h(\text{Hell})=\text{Ifmm}$, $h(\text{l})=\text{z}$.
Then $h(\text{Hel})h(\text{l}) = \text{oooz}$ but $h(\text{Hell}) = \text{Ifmm}$. The axiom breaks.

**Mappings defined on strings of length 1 are guaranteed homomorphisms**, because a
single character has no substrings other than itself and $\varepsilon$.

## 2. Definitions

### A deliberately broken mapping

In [ ]:
BROKEN = {'Hel': 'ooo', 'Hell': 'Ifmm', 'l': 'z'}

def h_broken(s):
    return BROKEN.get(s, '?' * len(s))

### A checker for the axiom

In [ ]:
def is_homomorphism(h, samples):
    """Check h(xy) == h(x)h(y) over every split of every sample."""
    bad = []
    for w in samples:
        for i in range(len(w)+1):
            if h(w) != h(w[:i]) + h(w[i:]):
                bad.append((w, i, h(w), h(w[:i]) + h(w[i:])))
    return (not bad), bad[:3]

<!-- nav-strip -->

---

&larr;&nbsp;[Ch3&nbsp;10.&nbsp;String and Language Homomorphisms](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter3-Star/Concept-Homomorphisms/Concept-Homomorphisms.ipynb) &nbsp;&middot;&nbsp; [**Chapter 3** index](https://github.com/ganeshutah/Jove/blob/master/Chapter3-Star/README.md) &nbsp;&middot;&nbsp; [Ch3&nbsp;12.&nbsp;Theorem: $L^{](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter3-Star/Concept-Star-Star-Is-Star/Concept-Star-Star-Is-Star.ipynb)&nbsp;&rarr;

---

## 3. Tests

The broken map fails, and we can point at exactly where.

In [ ]:
ok, bad = is_homomorphism(h_broken, ['Hell'])
print("h_broken a homomorphism? :", ok)
for w, i, whole, split in bad:
    print("   split %r at %d : h(w)=%r but h(x)h(y)=%r" % (w, i, whole, split))
assert not ok
print()
print("h('Hel') + h('l') = 'oooz'  !=  h('Hell') = 'Ifmm'.  Conflict on a substring.")

A **per-character** map passes on everything we throw at it.

In [ ]:
hm = lambda x: chr((ord(x)+1) % 256)
h_char = lambda s: shomo(s, hm)
ok, bad = is_homomorphism(h_char, ['Hell', 'abc', '', 'x', 'Hello there'])
print("per-character map a homomorphism? :", ok)
assert ok
print("  Single characters have no substrings, so there is nothing to conflict.")

The exercise: is **reversal** a homomorphism?

In [ ]:
ok, bad = is_homomorphism(lambda s: s[::-1], ['abc', 'ab'])
print("reversal a homomorphism? :", ok)
print("   first failure :", bad[0] if bad else None)
assert not ok
print("   It swaps the order of the pieces -- an ANTI-homomorphism.")

## 4. Exercises


1. Define $f$ mapping each letter two higher (mod 26) **and** `ab` to `c`. Use
   `is_homomorphism` to show it fails, and identify the conflicting split.
2. Can a homomorphism map a single character to a *longer* string? To $\varepsilon$?
3. Why does the book insist on lambdas over characters in `shomo`/`lhomo`?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 254 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter3-Star/Concept-Crux-Of-Homomorphisms')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')